# 01A: Exploring the Data Lake

Generic tooling to inspect the CICCADA data lake at three levels: **Storage** (S3 files), **Catalog** (Glue tables + schema), **Data** (actual rows). Reusable for any new data source.

**Three zoom levels:**

| Level | Question | Tool | Cost |
|---|---|---|---|
| **Storage** | What files physically exist? | `s3_ls()` (boto3) | free |
| **Catalog** | What's registered as a queryable table? | `databases()`, `tables()`, `describe()` (Glue) | free |
| **Data** | What do the values look like? | `aq()` (Athena) or `dread()` (DuckDB) | small / local |

Note: If anything says *"token expired"*, run `aws sso login --profile ciccada` in a terminal and re-run the cell.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
sys.path.insert(0, str(pathlib.Path('lib').resolve()))

from shared.aws_config import *          # aq, dread, s3_ls, databases, tables
from shared.ciccada_config import SA, SAI, TABLES
import pandas as pd

## [Level 1] Storage: the physical files in S3

In [ ]:
s3_ls()

In [ ]:
s3_ls('Trino-Warehouse/solar_analytics/')

## [Level 2] Catalog: what's queryable, and its schema

Glue is the index that lets us write SQL. These calls are (metadata only).

In [ ]:
databases()

In [ ]:
# Every table in every database, in one view.
all_tabs = pd.concat([tables(d) for d in databases()['Database']], ignore_index=True)
all_tabs

In [ ]:
# Peek at any table's schema (works on Iceberg via SELECT * LIMIT 1)
# table = TABLES['conformance_voltvar']   # 'conformance_voltvar_v2'
# or directly
# table = "meta_up23c"
table = "ts"
table_df = aq(f'SELECT * FROM {table} LIMIT 5', database=SAI)

In [ ]:
table_df.columns

In [ ]:
table_df

In [ ]:
table = "meta_up23c"  
table_df = aq(f'SELECT * FROM {table} LIMIT 5', database="SAI")
table_df

In [ ]:
table = "solar"  
table_df = aq(f'SELECT * FROM {table} LIMIT 5', database="bom_nci")
table_df

## [Level 3] Data peek at actual rows

Two engines, same files. 

1. Use `aq()` (Athena) for normal SQL
2. use `dread()` (DuckDB, reads the Parquet file directly) for quick peeks or when Glue's description is wrong.

**Cost:** on the big `ts` table, always filter on `year`/`month`/`is_pv` and never sort on a fresh peek.

In [ ]:
aq("SELECT * FROM ts WHERE is_pv = True AND year = 2024 AND month = 1 LIMIT 5", database=SAI)

In [ ]:
pd.set_option('display.max_columns', None)
aq("SELECT * FROM structured_data_v2 WHERE year = 2024 AND month = 1 LIMIT 5", database=SAI)

## If Athena chokes: read the Parquet directly with DuckDB

Some results tables have *schema drift*. The Glue description disagrees with the file (e.g. a column the file stores as integer but Glue calls a double). This is common.

Athena won't be able to read it, so DuckDB reads the file's own schema and just works.

The S3 path comes from the error message, or from `s3_ls()`.

In [ ]:
# dread('s3://project-ciccada/Trino-Warehouse/solar_analytics/conformance_voltvar_v2/data/*.parquet')

## Recipe to explore any new data source in future

1. **`s3_ls('<prefix>/')`**: See what physically exists and how it's "foldered".
2. **`databases()` / `tables(db)`**: Check if it is registered in Glue. If yes, can use SQL.
3. **`describe('<table>', db)`**: Learn its columns.
4. **`aq('SELECT ... LIMIT 5')`**: Peek at values (filter on partitions if it's big).
5. Note: Not in Glue, or Glue is wrong?: Point **`dread('s3://.../*.parquet')`** straight at the files.